# Running Conditional Simulation Compute

This notebook demonstrates a complete conditional simulation workflow:

1. Load downhole assay data as a **PointSet**
2. Define a normal-score **Variogram** model for spatial correlation
3. Create a target **Regular3DGrid** to simulate onto
4. Run **conditional turning-band simulation** using Evo Compute
5. Inspect summary statistics, quantiles and risk measures

## Requirements

You must have a Seequent account with the Evo entitlement. You'll need:
- The client ID of your Evo application
- The callback/redirect URL of your Evo application

To obtain these credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens).

The full task specification is documented in the
[Conditional simulation workflow API reference](https://developer.seequent.com/docs/api/geostatistics-task/tasks#conditional-simulation-workflow).

## 1. Authentication

Authenticate using the `ServiceManagerWidget` which handles OAuth login and workspace selection.

In [ ]:
from pathlib import Path

from evo.notebooks import ServiceManagerWidget

# Replace with your Evo app credentials
client_id = "<your-client-id>"
redirect_url = "<your-redirect-url>"

cache_location = Path.home() / ".evo-cache" / "running-conditional-simulation"

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    redirect_url=redirect_url,
    cache_location=cache_location,
).login()

In [ ]:
# Load the widgets extension for rich HTML display
%load_ext evo.widgets

## 2. Load and Create PointSet from Drill Hole Data

The WP_assay.csv dataset contains:
- **8,332 sample points** from 55 downholes
- **Coordinates**: X, Y, Z (EPSG:32650 - UTM Zone 50N)
- **Attributes**: Hole ID, CU_pct (copper %), AU_gpt (gold g/t), DENSITY

The conditioning data for a conditional simulation must be a `pointset` object with a
single continuous attribute to simulate.

In [ ]:
import pandas as pd

# Load the sample assay data
input_file = "sample-data/WP_assay.csv"
df = pd.read_csv(input_file)

print(f"Loaded {len(df)} sample points from {df['Hole ID'].nunique()} downholes")

# Select only the columns we need and rename coordinates
df = df[["X", "Y", "Z", "CU_pct"]].rename(columns={"X": "x", "Y": "y", "Z": "z"})

# Remove rows with null values - compute tasks require non-null values
original_count = len(df)
df = df.dropna().reset_index(drop=True)
removed_count = original_count - len(df)
if removed_count > 0:
    print(f"\nRemoved {removed_count} rows with null values")
print(f"Remaining: {len(df)} sample points")

# Verify no nulls remain
assert df.isna().sum().sum() == 0, "DataFrame still contains null values!"

print("\nSpatial extent:")
print(f"  X: {df['x'].min():.1f} to {df['x'].max():.1f} ({df['x'].max() - df['x'].min():.1f}m)")
print(f"  Y: {df['y'].min():.1f} to {df['y'].max():.1f} ({df['y'].max() - df['y'].min():.1f}m)")
print(f"  Z: {df['z'].min():.1f} to {df['z'].max():.1f} ({df['z'].max() - df['z'].min():.1f}m)")
print("\nCopper (CU_pct) statistics:")
print(f"  Mean: {df['CU_pct'].mean():.3f}%, Variance: {df['CU_pct'].var():.3f}")
print(f"  Min: {df['CU_pct'].min():.3f}%, Max: {df['CU_pct'].max():.3f}%")
df.head()

In [ ]:
from datetime import datetime, timezone

from evo.objects.typed import EpsgCode, PointSet, PointSetData

pointset_data = PointSetData(
    name=f"WP Drill Hole Assays {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
    description="Copper assay data from 55 downholes, used to condition the simulation",
    locations=df[["x", "y", "z", "CU_pct"]].copy(),
    coordinate_reference_system=EpsgCode(32650),  # UTM Zone 50N
)

pointset = await PointSet.create(manager, pointset_data)
print(f"Created pointset with {pointset.num_points} points")

In [ ]:
# Display the pointset with rich HTML formatting
pointset

In [ ]:
cu_attribute = pointset.attributes["CU_pct"]
source_attribute = f"locations.attributes[?key=='{cu_attribute.key}']"

print(f"Source attribute expression: {source_attribute}")

## 3. Create a Normal-Score Variogram Model

Conditional simulation works in **Gaussian (normal-score) space**. The task transforms
the source values through a continuous distribution, simulates in Gaussian space, then
back-transforms the realisations to data units.

Because of this, the variogram supplied to the task must model the covariance of the
**normal-score transformed** data:

- `modelling_space="normalscore"`
- `sill=1.0` (the variance of a standard Gaussian)
- `nugget` plus all structure contributions must sum to the sill

The anisotropy is aligned with the dominant NNE-SSW trend of the downholes:
- Dip azimuth: 15 degrees (strike direction)
- Dip: 70 degrees (steep dip to the east)
- Pitch: 0 degrees

In [ ]:
from evo.objects.typed import (
    Ellipsoid,
    EllipsoidRanges,
    Rotation,
    SphericalStructure,
    Variogram,
    VariogramData,
)

variogram_data = VariogramData(
    name=f"CU_pct Variogram {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
    sill=1.0,  # Standard Gaussian variance
    nugget=0.10,  # nugget(0.10) + 0.30 + 0.60 = 1.0
    is_rotation_fixed=True,  # All structures share the same rotation
    modelling_space="normalscore",  # Required for conditional simulation
    data_variance=1.0,
    attribute="CU_pct",
    structures=[
        # Short-range structure
        SphericalStructure(
            contribution=0.30,
            anisotropy=Ellipsoid(
                ranges=EllipsoidRanges(major=80.0, semi_major=60.0, minor=40.0),
                rotation=Rotation(dip=70.0, dip_azimuth=15.0, pitch=0.0),
            ),
        ),
        # Long-range structure
        SphericalStructure(
            contribution=0.60,
            anisotropy=Ellipsoid(
                ranges=EllipsoidRanges(major=250.0, semi_major=180.0, minor=100.0),
                rotation=Rotation(dip=70.0, dip_azimuth=15.0, pitch=0.0),
            ),
        ),
    ],
)

variogram = await Variogram.create(manager, variogram_data)
print(f"Created variogram: {variogram.name}")

In [ ]:
# Display the variogram with rich HTML formatting
variogram

In [ ]:
# Get variogram curves for the three principal directions
import plotly.graph_objects as go

major, semi_major, minor = variogram.get_principal_directions()

fig = go.Figure()
for curve, label, color in [
    (minor, "Minor", "blue"),
    (semi_major, "Semi-major", "green"),
    (major, "Major", "red"),
]:
    fig.add_trace(
        go.Scatter(
            x=curve.distance,
            y=curve.semivariance,
            name=f"{label} (range={curve.range_value:.0f}m)",
            line=dict(color=color, width=2),
        )
    )

fig.add_hline(y=variogram.nugget, line_dash="dash", line_color="gray", annotation_text="Nugget")
fig.add_hline(y=variogram.sill, line_dash="dash", line_color="black", annotation_text="Sill")

fig.update_layout(
    title="CU_pct Normal-Score Variogram Model - Principal Directions",
    xaxis_title="Lag Distance (m)",
    yaxis_title="Semivariance in Gaussian space",
    template="plotly_white",
)
fig.show()

## 4. Define the Search Neighbourhood

The search neighbourhood controls which conditioning samples influence each simulated
location.

In [ ]:
from evo.compute.tasks import SearchNeighborhood

var_ellipsoid = variogram.get_ellipsoid()
search_ellipsoid = var_ellipsoid.scaled(2.0)

search = SearchNeighborhood(
    ellipsoid=search_ellipsoid,
    max_samples=24,  # Maximum conditioning samples per location
    min_samples=4,  # Minimum samples required
)

print(f"Variogram ellipsoid: major={var_ellipsoid.ranges.major}m")
print(f"Search ellipsoid (2x): major={search_ellipsoid.ranges.major}m")

## 5. Create the Target Grid

Conditional simulation evaluates onto a `regular-3d-grid` or `regular-masked-3d-grid`. The task creates new
attributes on this grid for the summary statistics, quantiles and saved realisations.

In [ ]:
from evo.objects.typed import Point3, Regular3DGrid, Regular3DGridData, Size3d, Size3i

# Grid geometry covering the drill hole extent - reused for the scenarios below
grid_origin = Point3(x=444750, y=492850, z=2350)
grid_size = Size3i(nx=25, ny=35, nz=22)
grid_cell_size = Size3d(dx=40.0, dy=40.0, dz=40.0)

grid = await Regular3DGrid.create(
    manager,
    Regular3DGridData(
        name=f"CU Conditional Simulation Target Grid {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
        description="Target grid for conditional simulation of copper grades",
        origin=grid_origin,
        size=grid_size,
        cell_size=grid_cell_size,
        coordinate_reference_system=EpsgCode(32650),
    ),
)

print(f"Created grid: {grid.name}")
print(f"Cells: {grid_size.nx * grid_size.ny * grid_size.nz:,}")

In [ ]:
# Display the grid metadata
grid

## 6. Configure the Conditional Simulation

`ConSimParameters` drives the whole workflow. The main groups of settings are:

| Setting | Purpose |
| --- | --- |
| `distribution` | Tail extrapolation for the normal-score transformation |
| `block_discretization` | Sub-block discretization for support correction |
| `number_of_lines` | Turning bands used by the simulator |
| `number_of_simulations` | How many realisations to generate |
| `number_of_simulations_to_save` | How many realisations to publish to the grid |
| `location_wise_quantiles` | Quantiles computed per block across realisations |
| `probability_above_cutoff` | P(grade > cutoff) per block |
| `mean_above_cutoff` | E[grade \| grade > cutoff] per block |
| `perform_validation` | Generate a validation report and dashboard link |

Tail extrapolation extends the sample distribution beyond the observed data so the
normal-score back-transform is defined over the full Gaussian range. When `distribution`
is supplied, the task requires **both** tails: `upper.max` must be greater than the
maximum value in the data, and `lower.min` must be less than the minimum. Omit
`distribution` entirely to let the service derive the distribution from the data.


In [ ]:
from evo.compute.tasks.geostatistics.conditioned_simulator import (
    BlockDiscretization,
    ConSimParameters,
    DistributionParams,
    LowerTailParams,
    ReportContext,
    ReportMeanThresholds,
    TailExtrapolationParams,
    UpperTailParams,
    ValidationReportContextItem,
)

cu_min = float(df["CU_pct"].min())
cu_max = float(df["CU_pct"].max())

upper_max = cu_max * 1.5
lower_min = cu_min - max(0.01, abs(cu_min) * 0.1)

params = ConSimParameters(
    source_object=pointset,
    source_attribute=source_attribute,
    target_object=grid,
    variogram_model=variogram,
    neighborhood=search,
    distribution=DistributionParams(
        tail_extrapolation=TailExtrapolationParams(
            upper=UpperTailParams(power=0.5, max=upper_max),
            lower=LowerTailParams(power=0.5, min=lower_min),
        ),
    ),
    kriging_method="simple",  # Simple kriging is standard for Gaussian simulation
    block_discretization=BlockDiscretization(nx=3, ny=3, nz=2),
    number_of_lines=500,
    number_of_simulations=20,
    number_of_simulations_to_save=5,
    random_seed=38239342,
    location_wise_quantiles=[0.1, 0.5, 0.9],
    probability_above_cutoff=[0.5, 1.0],
    mean_above_cutoff=[0.5, 1.0],
    perform_validation=True,
    report_context=ReportContext(
        title="WP Copper Conditional Simulation",
        details=[
            ValidationReportContextItem(label="Deposit", value="WP"),
            ValidationReportContextItem(label="Attribute", value="CU_pct"),
        ],
    ),
    report_mean_thresholds=ReportMeanThresholds(acceptable=5.0, marginal=10.0),
)

print(f"Simulating {params.number_of_simulations} realisations, saving {params.number_of_simulations_to_save}")
print(f"Tail extrapolation: {lower_min:.3f}% to {upper_max:.3f}% (data range {cu_min:.3f}% to {cu_max:.3f}%)")

### Optional: restrict the simulation with filters

The conditional simulation task can restrict the simulation to a subset of the target grid
and/or use only a subset of the conditioning data. Build the expression from a
`FilterCondition` (or the `AllOfFilter` / `AnyOfFilter` composites for AND / OR logic)
and pass it as `filter` (target grid) or `source_filter` (conditioning data):

```python
from evo.compute.tasks import Filter, FilterCondition

params = ConSimParameters(
    ...,
    # Only simulate blocks whose 'domain' category is LMS1 or LMS2:
    filter=Filter(
        where=FilterCondition(
            attribute="domain",
            operator="in",
            values=["LMS1", "LMS2"],
        ),
    ),
    # Only condition on samples above a minimum grade:
    source_filter=Filter(
        where=FilterCondition(
            attribute="CU_pct",
            operator="greater_than",
            threshold=0.05,
        ),
    ),
)
```

Membership operators (`in`, `not_in`) take `values`; the numeric operators (`equal`,
`not_equal`, `greater_than`, `greater_than_or_equal_to`, `less_than`,
`less_than_or_equal_to`) take a single `threshold`.

## 7. Run the Conditional Simulation Task

Submit and run the conditional simulation task using Evo Compute.

In [ ]:
from evo.compute.tasks import run

print("Submitting conditional simulation task...")
result = await run(manager, params)

print("Conditional simulation complete!")

In [ ]:
# Display the result (pretty-printed in Jupyter)
result

## 8. Inspect the Simulation Outputs

The task writes several families of attributes onto the target grid:

- **Summary statistics** - mean, variance, min and max across all realisations
- **Quantiles** - one attribute per requested quantile
- **Risk measures** - probability above cutoff and mean above cutoff
- **Realisations** - the saved simulations, in data units and normal-score space

In [ ]:
print(f"Target grid: {result.target_name}")

print("\nSummary attributes:")
print(f"  Mean:     {result.summary_attributes.mean.name}")
print(f"  Variance: {result.summary_attributes.variance.name}")
print(f"  Min:      {result.summary_attributes.min.name}")
print(f"  Max:      {result.summary_attributes.max.name}")

if result.quantile_attributes:
    print("\nQuantile attributes:")
    for quantile_attr in result.quantile_attributes:
        print(f"  P{quantile_attr.quantile * 100:.0f}: {quantile_attr.name}")

if result.probability_above_cutoff_attributes:
    print("\nProbability above cutoff:")
    for cutoff_attr in result.probability_above_cutoff_attributes:
        print(f"  > {cutoff_attr.cutoff}%: {cutoff_attr.name}")

if result.mean_above_cutoff_attributes:
    print("\nMean above cutoff:")
    for cutoff_attr in result.mean_above_cutoff_attributes:
        print(f"  > {cutoff_attr.cutoff}%: {cutoff_attr.name}")

if result.simulations_attribute:
    print(f"\nSaved realisations: {result.simulations_attribute.name}")

In [ ]:
# Validation compares the mean of the input samples against the mean of the simulations
if result.validation_summary:
    reference_mean = result.validation_summary.reference_mean
    simulated_mean = result.validation_summary.mean
    difference = 100.0 * (simulated_mean - reference_mean) / reference_mean
    print(f"Reference (sample) mean: {reference_mean:.4f}%")
    print(f"Simulated mean:          {simulated_mean:.4f}%")
    print(f"Difference:              {difference:+.2f}%")

if result.dashboard_url:
    print(f"\nValidation dashboard: {result.dashboard_url}")

In [ ]:
# Pull the simulation summary statistics back as a DataFrame
results_df = await result.to_dataframe()

print(f"Simulated {len(results_df):,} cells")
results_df.describe()

In [ ]:
# Compare the P10 / P50 / P90 grade distributions to visualise uncertainty
quantile_columns = {
    f"P{quantile_attr.quantile * 100:.0f}": quantile_attr.name for quantile_attr in result.quantile_attributes
}

fig = go.Figure()
for label, column in quantile_columns.items():
    if column in results_df.columns:
        fig.add_trace(go.Histogram(x=results_df[column].dropna(), name=label, opacity=0.6, nbinsx=60))

fig.update_layout(
    title="Simulated CU_pct - quantile distributions across the grid",
    xaxis_title="CU_pct (%)",
    yaxis_title="Number of cells",
    barmode="overlay",
    template="plotly_white",
)
fig.show()

## 9. View Objects in Evo

Generate a viewer URL to inspect the conditioning data and the simulated grid together.

In [ ]:
from evo.widgets import get_viewer_url_for_objects

viewer_url = get_viewer_url_for_objects(manager, [pointset, grid])
print(f"View in Evo Viewer: {viewer_url}")

## 10. Publish the Results as a Block Model

The three inputs to the simulation each contribute to a single block model, which is the
form mine planning and reporting tools consume:

| Source | Contribution |
| --- | --- |
| **Target grid** | Block geometry (origin, block counts, block size) and **every** simulated attribute - mean, variance, min, max, quantiles, cutoff statistics and any saved realisations |
| **PointSet** | Distance from each block centre to the nearest conditioning assay |
| **Variogram** | The modelled major range, used to normalise that distance into a confidence classification |

This uses `result` from the single simulation run above. The well-known outputs are given
friendly aliases (`sim_mean`, `sim_p10`, `prob_above_0.5`, ...); anything else keeps the
name the service generated.

Block models live in the Block Model Service rather than as plain geoscience objects, so
`BlockModel.create_regular` creates both the block model and the geoscience object that
references it. Cell data is matched to blocks through the `i`, `j`, `k` index columns,
which follow the same x-fastest ordering as the grid's cells.


In [ ]:
import numpy as np

from evo.blockmodels.typed import Units
from evo.objects.typed import BlockModel, RegularBlockModelData

i_idx, j_idx, k_idx = np.indices((grid_size.nx, grid_size.ny, grid_size.nz))
block_df = pd.DataFrame(
    {
        "i": i_idx.ravel(order="F"),
        "j": j_idx.ravel(order="F"),
        "k": k_idx.ravel(order="F"),
    }
)

# --- From the target grid: every simulated attribute ---
# Friendly aliases for the well-known outputs; anything else keeps its service-generated name.
column_aliases = {
    result.summary_attributes.mean.name: "sim_mean",
    result.summary_attributes.variance.name: "sim_variance",
    result.summary_attributes.min.name: "sim_min",
    result.summary_attributes.max.name: "sim_max",
}
column_aliases.update({q.name: f"sim_p{q.quantile * 100:.0f}" for q in result.quantile_attributes})
column_aliases.update({a.name: f"prob_above_{a.cutoff:g}" for a in result.probability_above_cutoff_attributes})
column_aliases.update({a.name: f"mean_above_{a.cutoff:g}" for a in result.mean_above_cutoff_attributes})

simulation_df = results_df.reset_index(drop=True).rename(columns=column_aliases)

for column in simulation_df.select_dtypes(include="category").columns:
    simulation_df[column] = simulation_df[column].astype(str)

block_df = pd.concat([block_df, simulation_df], axis=1)
print(f"Carrying {len(simulation_df.columns)} simulation attributes onto the block model")

# --- From the pointset: distance from each block centre to the nearest assay sample ---
block_centres = np.column_stack(
    [
        grid_origin.x + (block_df["i"].to_numpy() + 0.5) * grid_cell_size.dx,
        grid_origin.y + (block_df["j"].to_numpy() + 0.5) * grid_cell_size.dy,
        grid_origin.z + (block_df["k"].to_numpy() + 0.5) * grid_cell_size.dz,
    ]
)
sample_xyz = df[["x", "y", "z"]].to_numpy(dtype=float)

nearest_sample = np.empty(len(block_df))
for start in range(0, len(block_df), 256):  # Chunked to bound peak memory
    chunk = block_centres[start : start + 256]
    squared = ((chunk[:, None, :] - sample_xyz[None, :, :]) ** 2).sum(axis=2)
    nearest_sample[start : start + 256] = np.sqrt(squared.min(axis=1))

# --- From the variogram: normalise the sample distance against the modelled range ---
major_range = var_ellipsoid.ranges.major
block_df["nearest_sample_m"] = nearest_sample
block_df["range_fraction"] = nearest_sample / major_range
block_df["classification"] = np.select(
    [block_df["range_fraction"] <= 0.5, block_df["range_fraction"] <= 1.0],
    ["Measured", "Indicated"],
    default="Inferred",
)

max_distance_from_data = major_range
if max_distance_from_data is not None:
    beyond_data = block_df["nearest_sample_m"] > max_distance_from_data
    block_df.loc[beyond_data, simulation_df.columns] = np.nan
    block_df.loc[beyond_data, "classification"] = "Unclassified"
    print(f"Blanked {beyond_data.sum():,} of {len(block_df):,} blocks beyond {max_distance_from_data:.0f}m from data")

block_model = await BlockModel.create_regular(
    manager,
    RegularBlockModelData(
        name=f"CU Conditional Simulation Block Model {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
        description=(
            f"Conditional simulation of CU_pct. Conditioned on '{pointset.name}', "
            f"variogram '{variogram.name}', evaluated on grid '{grid.name}'."
        ),
        origin=grid_origin,
        n_blocks=grid_size,
        block_size=grid_cell_size,
        cell_data=block_df,
        coordinate_reference_system="EPSG:32650",
        size_unit_id=Units.METRES,
        units={"nearest_sample_m": Units.METRES},
    ),
)

print(f"\nCreated block model: {block_model.name}")
print(f"Blocks: {len(block_df):,}, columns: {len(block_df.columns)}")
for column in block_df.columns:
    print(f"  {column}")
print("\nClassification breakdown:")
print(block_df["classification"].value_counts())
block_model

## 11. Running Multiple Simulation Scenarios

Multiple simulations can be submitted together and run concurrently. Here we compare
**point support** (no sub-block discretization) against **block support** (each cell
discretized into 3 x 3 x 2 sub-cells), which is the main control on the variance of the
simulated block grades.

Each scenario writes to its own grid so the output attributes stay separate. These
scenarios are for sensitivity analysis only - the block model above is built from the
single simulation run in section 7.


In [ ]:
scenarios = {
    "point-support": BlockDiscretization(nx=1, ny=1, nz=1),
    "block-support": BlockDiscretization(nx=3, ny=3, nz=2),
}

scenario_grids = {}
parameter_sets = []

for label, discretization in scenarios.items():
    scenario_grid = await Regular3DGrid.create(
        manager,
        Regular3DGridData(
            name=f"CU Conditional Simulation ({label}) {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
            description=f"Conditional simulation target grid - {label}",
            origin=grid_origin,
            size=grid_size,
            cell_size=grid_cell_size,
            coordinate_reference_system=EpsgCode(32650),
        ),
    )
    scenario_grids[label] = scenario_grid
    parameter_sets.append(
        ConSimParameters(
            source_object=pointset,
            source_attribute=source_attribute,
            target_object=scenario_grid,
            variogram_model=variogram,
            neighborhood=search,
            block_discretization=discretization,
            number_of_simulations=10,
            number_of_simulations_to_save=3,
        )
    )
    print(f"Prepared scenario '{label}' targeting {scenario_grid.name}")

In [ ]:
print(f"Submitting {len(parameter_sets)} conditional simulation tasks...")
results = await run(manager, parameter_sets, preview=True)

print(f"\nAll {len(results)} scenarios completed!")
results

In [ ]:
# Compare the simulated variance between the two support scenarios
for label, scenario_result in zip(scenarios, results):
    scenario_df = await scenario_result.to_dataframe()
    variance_column = scenario_result.summary_attributes.variance.name
    mean_column = scenario_result.summary_attributes.mean.name
    print(f"{label}:")
    print(f"  Mean of simulated means:     {scenario_df[mean_column].mean():.4f}%")
    print(f"  Mean of simulated variances: {scenario_df[variance_column].mean():.4f}")